In [27]:
import os
import zipfile
import pandas as pd

In [29]:
if not os.path.isdir("/content/coco-locations"):
    with zipfile.ZipFile("/content/coco-locations.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

In [30]:
data = pd.read_csv("/content/COCO-locations.csv")
data.head()

,id,cap,url,background
0,555597,A black and white image of a city street in th...,http://images.cocodataset.org/val2017/00000055...,"['1960s', 'street', 'city']"
1,9378,Adult man displaying abilities using flying ye...,http://images.cocodataset.org/val2017/00000000...,['ability']
2,572678,An upscale living area containing white and gl...,http://images.cocodataset.org/val2017/00000057...,"['accent', 'area', 'furniture']"
3,462614,A bathroom has red walls with yellow accents.,http://images.cocodataset.org/val2017/00000046...,"['accent', 'wall']"
4,308466,"A bathroom containing a toilet, sink and batht...",http://images.cocodataset.org/val2017/00000030...,"['accessory', 'shower', 'bathtub']"


In [35]:
!pip install scikit-multilearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 3.8 MB/s eta 0:00:00


In [36]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, accuracy_score
import ast
import re
import random
from collections import Counter

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Load the dataset
data = pd.read_csv("/content/COCO-locations.csv")

# Define a function to safely parse the background column
def safe_literal_eval(val):
    try:
        return ast.literal_eval(val)
    except (SyntaxError, ValueError):
        # Try to fix common errors
        try:
            # Fix missing closing quote
            if "'" in val and val.count("'") % 2 != 0:
                val = val + "'"

            # Fix missing closing bracket
            if val.startswith("[") and not val.endswith("]"):
                val = val + "]"

            return ast.literal_eval(val)
        except:
            # If all attempts fail, return an empty list
            print(f"Warning: Could not parse value: {val}")
            return []

# Apply the safe parsing function
data['background'] = data['background'].apply(safe_literal_eval)

# Get all unique location tags
all_tags = set()
for tags in data['background']:
    for tag in tags:
        all_tags.add(tag)
print(f"Number of unique location tags: {len(all_tags)}")
print(f"Example tags: {list(all_tags)[:10]}")

# Analyze label distribution
tag_counts = Counter()
for tags in data['background']:
    for tag in tags:
        tag_counts[tag] += 1

print("Most common tags:")
for tag, count in tag_counts.most_common(10):
    print(f"{tag}: {count}")

print("\nLeast common tags:")
for tag, count in tag_counts.most_common()[-10:]:
    print(f"{tag}: {count}")

# Only keep tags that appear at least 5 times
min_tag_count = 5
common_tags = {tag for tag, count in tag_counts.items() if count >= min_tag_count}
print(f"Number of tags appearing at least {min_tag_count} times: {len(common_tags)}")

# Filter the background lists to only include common tags
data['background_filtered'] = data['background'].apply(lambda tags: [tag for tag in tags if tag in common_tags])

# Remove samples with no tags after filtering
data = data[data['background_filtered'].apply(len) > 0].reset_index(drop=True)
print(f"Remaining samples after filtering: {len(data)}")

# Create multi-label encoding for the filtered background tags
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(data['background_filtered'])
print(f"Shape of encoded labels: {y_encoded.shape}")

# Split the data with stratification
from skmultilearn.model_selection import iterative_train_test_split

# Convert to array format for iterative_train_test_split
X = np.array([[i] for i in range(len(data))])
y = y_encoded

# Perform iterative train/test split to maintain label distribution
X_train_idx, y_train, X_test_idx, y_test = iterative_train_test_split(X, y, test_size=0.2)

# Get actual data using indices
X_train = data.iloc[X_train_idx.flatten()]['cap'].values
X_test = data.iloc[X_test_idx.flatten()]['cap'].values

# Define the dataset class with weighted loss handling
class LocationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

# Load RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

# Calculate class weights for weighted loss
pos_weight = torch.tensor(
    [(len(data) - sum(y_encoded[:, i])) / max(sum(y_encoded[:, i]), 1) for i in range(y_encoded.shape[1])],
    dtype=torch.float
)

# Prepare datasets and dataloaders
train_dataset = LocationDataset(X_train, y_train, tokenizer)
test_dataset = LocationDataset(X_test, y_test, tokenizer)

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

# Set up training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)
pos_weight = pos_weight.to(device)

# Use weighted binary cross-entropy loss to handle class imbalance
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=1e-5)  # Lower learning rate

# Training function with custom loss
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)

# Evaluation function with lower threshold
def evaluate(model, dataloader, device, threshold=0.3):  # Lower threshold for predictions
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.sigmoid(logits) > threshold

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    predictions = np.array(predictions, dtype=int)
    true_labels = np.array(true_labels, dtype=int)

    accuracy = accuracy_score(true_labels.flatten(), predictions.flatten())
    f1 = f1_score(true_labels, predictions, average='micro')

    return accuracy, f1, predictions, true_labels

# Training loop with more epochs
num_epochs = 5

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    avg_loss = train_epoch(model, train_dataloader, optimizer, criterion, device)
    print(f"Average training loss: {avg_loss:.4f}")

    accuracy, f1, _, _ = evaluate(model, test_dataloader, device)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test F1 Score: {f1:.4f}")
    print("-" * 50)

# Save the model
model.save_pretrained("/content/roberta_location_classifier")
tokenizer.save_pretrained("/content/roberta_location_classifier")

# Function to predict location for new text with lower threshold
def predict_location(text, model, tokenizer, mlb, device, threshold=0.3):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.sigmoid(logits)

        # Get top 5 predictions if none are above threshold
        if not torch.any(probs > threshold):
            top_indices = torch.topk(probs, min(5, probs.shape[1])).indices[0].cpu().numpy()
            predicted_tags = [mlb.classes_[i] for i in top_indices]
        else:
            predictions = (probs > threshold).cpu().numpy()
            predicted_tags = []
            for i, pred in enumerate(predictions[0]):
                if pred:
                    predicted_tags.append(mlb.classes_[i])

    return predicted_tags, probs.cpu().numpy()[0]

# Test on some examples from the test set
print("Predictions on test examples:")
test_examples = []
test_indices = np.random.choice(range(len(X_test)), 10, replace=False)

for i, idx in enumerate(test_indices):
    caption = X_test[idx]
    true_tags = mlb.inverse_transform(y_test[idx].reshape(1, -1))[0]

    predicted_tags, probs = predict_location(caption, model, tokenizer, mlb, device)

    # Get tag probabilities
    tag_probs = [(tag, probs[list(mlb.classes_).index(tag)]) for tag in predicted_tags]
    tag_probs.sort(key=lambda x: x[1], reverse=True)

    test_examples.append({
        'caption': caption,
        'true_tags': list(true_tags),
        'predicted_tags': predicted_tags
    })

    print(f"Example {i+1}:")
    print(f"Caption: {caption}")
    print(f"True tags: {list(true_tags)}")
    print(f"Predicted tags: {predicted_tags}")
    print(f"Top tag probabilities: {tag_probs}")
    print("-" * 50)

# Custom examples
custom_texts = [
    "A modern kitchen with stainless steel appliances and marble countertops.",
    "A busy street in downtown with people walking on sidewalks.",
    "A beautiful beach with palm trees and white sand.",
    "A cozy living room with a fireplace and bookshelves.",
    "An office space with computers and ergonomic chairs.",
    "A playground with swings and a slide in a park.",
    "A hospital waiting room with chairs and a reception desk.",
    "A classroom with desks and a whiteboard.",
    "A train station platform with people waiting.",
    "A garden with colorful flowers and a small pond."
]

print("Predictions on custom examples:")
custom_examples = []
for i, text in enumerate(custom_texts):
    predicted_tags, probs = predict_location(text, model, tokenizer, mlb, device)

    # Get tag probabilities
    tag_probs = [(tag, probs[list(mlb.classes_).index(tag)]) for tag in predicted_tags]
    tag_probs.sort(key=lambda x: x[1], reverse=True)

    custom_examples.append({
        'caption': text,
        'predicted_tags': predicted_tags
    })

    print(f"Example {i+1}:")
    print(f"Caption: {text}")
    print(f"Predicted tags: {predicted_tags}")
    print(f"Top tag probabilities: {tag_probs}")
    print("-" * 50)

# Save results to CSV files
test_df = pd.DataFrame(test_examples)
test_df.to_csv("/content/test_predictions.csv", index=False)

custom_df = pd.DataFrame(custom_examples)
custom_df.to_csv("/content/custom_predictions.csv", index=False)

Number of unique location tags: 3674
Example tags: ['tye', 'pot', 'shoe', 'berry', 'patchwork', 'past', 'duck', 'citizen', 'softball', 'hand']
Most common tags:
table: 1204
street: 1132
plate: 907
top: 897
field: 888
tennis: 779
front: 771
room: 704
building: 699
water: 646

Least common tags:
encloseur: 1
enjoy: 1
airy: 1
petting: 1
potty: 1
party: 1
snowboard: 1
world: 1
trolly: 1
bed]: 1
Number of tags appearing at least 5 times: 1142
Remaining samples after filtering: 23793
Shape of encoded labels: (23793, 1142)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda
Epoch 1/5
Average training loss: 1.3973
Test Accuracy: 0.0177
Test F1 Score: 0.0037
--------------------------------------------------
Epoch 2/5
Average training loss: 1.2049
Test Accuracy: 0.1316
Test F1 Score: 0.0042
--------------------------------------------------
Epoch 3/5
Average training loss: 1.0348
Test Accuracy: 0.2929
Test F1 Score: 0.0051
--------------------------------------------------
Epoch 4/5
Average training loss: 0.8979
Test Accuracy: 0.4042
Test F1 Score: 0.0061
--------------------------------------------------
Epoch 5/5
Average training loss: 0.7847
Test Accuracy: 0.5097
Test F1 Score: 0.0073
--------------------------------------------------
Predictions on test examples:
Example 1:
Caption: A young man playing out side with a disc.
True tags: ['side']
Predicted tags: ['air', 'airplane', 'amount', 'animal', 'apartment', 'apple', 'area', 'arm', 'back', 'background', 'ball', 'bare', 'base', 'baseball', 'basketball', 'bat', 'bath', 'beach', 'bed'